# Demo 1 — Tree of Thoughts on Game of 24
## Session 6: Advanced Prompt Engineering

> **MLflow as the tape recorder.** Every prompt sent, every response returned, every tool call, every latency and token count — captured automatically by `mlflow.openai.autolog()` and the `@mlflow.trace` decorators below. Run the cells, then open the MLflow UI to replay every input and output the model saw.

**Goal:** Show why **search > sampling** for branching problems where one wrong commit kills the chain.

**Story arc:**
1. Try the easy puzzle `[5, 5, 5, 5] → 24` with **zero-shot CoT** (one chain, no backup). Most attempts fail.
2. Switch to **Tree of Thoughts**: propose `k` next-step expressions per node, score each with a value-function LLM call, keep the top `beam` and expand. Watch the tree solve it.
3. Escalate to `[2, 7, 8, 8] → 24` (the harder Yao et al. puzzle).
4. Open MLflow UI to see the full search tree as a span tree + the `search_tree.json` artefact.

**Citation:** Yao et al. 2023, *Tree of Thoughts: Deliberate Problem Solving with Large Language Models* — [arXiv:2305.10601](https://arxiv.org/abs/2305.10601) (NeurIPS 2023). Headline number: GPT-4 **4% (CoT) → 74% (ToT)** on Game of 24.

> ⚠️ **Illustrative only.** We run on **2 puzzles** with a small number of trials. The 4 → 74 paper benchmark is the **citation**, not what we reproduce live. Treat the in-notebook success-rate printouts as illustrative of the *mechanism* (catastrophic early-commit, propose/evaluate/prune); not as a re-run of the paper.

**Dependencies:** `openai`, `mlflow >= 3.10`, `pandas`. Configure via env vars (see next cell) — a local MLflow server typically runs at:

```bash
mlflow server --host 127.0.0.1 --port 5000
```

**Cost budget for the full run:** roughly $0.10 of `gpt-4o` (two puzzles, ~30–50 calls each).

## Setup — env-var driven

The notebook reads every endpoint and credential from environment variables so it runs unchanged against local MLflow + OpenAI, Databricks, vLLM, Ollama, Azure, Together, or a CI sandbox.

```bash
export MLFLOW_TRACKING_URI=http://127.0.0.1:5000        # or Databricks workspace URL
export MLFLOW_EXPERIMENT=session6/demo_01_tot           # optional override
export OPENAI_BASE_URL=https://api.openai.com/v1           # OpenAI-compatible endpoint
export OPENAI_API_KEY=sk-...                                # leave unset to use MOCK mode
export MODEL=gpt-4o                                  # ToT needs a strong proposer/evaluator
```

No env vars set? The cell falls back to localhost MLflow + MOCK mode (no network).

**Local-by-default.** Notebook ships with `OPENAI_BASE_URL=http://localhost:11434/v1` and `MODEL=qwen2.5-coder:14b` (Ollama — ToT needs a strong proposer/evaluator). Set the env vars to point at OpenAI, Anthropic, vLLM, Together, etc. — any OpenAI-compatible endpoint.

**Cost tracking uses hypothetical local-LLM rates** (~$0.00005/1K tokens) so the cost-aware traces and scorers still produce meaningful numbers in the demos. Override the `PRICING` dict with your real per-1K rates for production.

In [ ]:
# -- Setup -----------------------------------------------------------------
import os
import re
import json
import time
import random
from typing import Optional

import pandas as pd
import mlflow
from mlflow.entities import SpanType
from openai import OpenAI

# ─── MLflow tracking ─────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI      = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
MLFLOW_TRACKING_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME")  # optional (Databricks/remote)
MLFLOW_TRACKING_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD")  # optional
MLFLOW_REGISTRY_URI      = os.getenv("MLFLOW_REGISTRY_URI", MLFLOW_TRACKING_URI)
MLFLOW_EXPERIMENT        = os.getenv("MLFLOW_EXPERIMENT", "session6/demo_01_tot")

if MLFLOW_TRACKING_USERNAME:
    os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
if MLFLOW_TRACKING_PASSWORD:
    os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_registry_uri(MLFLOW_REGISTRY_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

# ─── LLM (OpenAI-compatible: OpenAI, Azure, vLLM, Ollama, Together, etc.) ────
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "http://localhost:11434/v1")  # Ollama default
OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY", "ollama")  # Ollama ignores; any non-empty string works
MODEL    = os.getenv("MODEL", "qwen2.5-coder:14b")  # ToT needs strong proposer/evaluator; llama3.1:8b is a fallback

# Reproducibility default. Override via env for exploration.
# NOTE: ToT propose step intentionally overrides to >0.5 — search needs diversity.
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.0"))

# Per-1K-token pricing (USD). Update at each model release.
PRICING = {
    "gpt-4o":             {"in": 0.0025,   "out": 0.010},
    "gpt-4o-mini":        {"in": 0.00015,  "out": 0.0006},
    "o1-mini":            {"in": 0.003,    "out": 0.012},
    "claude-sonnet-4-6":  {"in": 0.003,    "out": 0.015},
    # Hypothetical local-LLM pricing (electricity + amortised GPU per 1K tokens — adjust to your infra)
    "qwen2.5-coder:14b":  {"in": 0.00005,  "out": 0.00005},
    "qwen2.5-coder:7b":   {"in": 0.00002,  "out": 0.00002},
    "llama3.1:8b":        {"in": 0.00002,  "out": 0.00002},
}

def price_of(model: str, usage) -> float:
    """Return USD cost for one chat call. Returns 0.0 in mock mode (no usage object).
    Unknown models fall back to hypothetical local-LLM rates (~$0.00005/1K)."""
    if usage is None:
        return 0.0
    p = PRICING.get(model, {"in": 0.00005, "out": 0.00005})  # hypothetical fallback
    return usage.prompt_tokens / 1000 * p["in"] + usage.completion_tokens / 1000 * p["out"]


def tag_cost_latency(latency_ms: float, cost_usd: float) -> None:
    """Attach cost + latency to the active span and root trace."""
    span = mlflow.get_current_active_span()
    if span:
        span.set_attribute("latency_ms", latency_ms)
        span.set_attribute("cost_usd", cost_usd)
    try:
        mlflow.update_current_trace(tags={
            "cost_usd":   f"{cost_usd:.6f}",
            "latency_ms": f"{latency_ms:.0f}",
        })
    except Exception:
        pass


client = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY or "no-key")

# ─── Turn MLflow into the tape recorder ──────────────────────────────────────
# Captures every chat.completions.create call — request, response, tokens, latency, cost.
mlflow.openai.autolog()

USE_MOCK = not OPENAI_API_KEY
print(f"MLflow:  {MLFLOW_TRACKING_URI}  (exp: {MLFLOW_EXPERIMENT})")
print(f"LLM:     {OPENAI_BASE_URL}  model={MODEL}  mock={USE_MOCK}  temperature={TEMPERATURE}")
print(f"UI:      open {MLFLOW_TRACKING_URI}  → Experiments → {MLFLOW_EXPERIMENT}")
print(f"NOTE: defaults to local Ollama (http://localhost:11434/v1). Set OPENAI_BASE_URL/OPENAI_API_KEY/MODEL to point at OpenAI/Anthropic/vLLM.")

## The Game of 24

Given four numbers, combine them with `+ - * /` (each used exactly once, parentheses allowed) to make exactly **24**.

Two puzzles for this demo:

| Numbers | Difficulty | Example solution |
|---|---|---|
| `[5, 5, 5, 5]` | Easy warm-up | `5 * 5 - 5 / 5 = 24` |
| `[2, 7, 8, 8]` | Hard (Yao et al.) | `(7 - 8/8) * (2 + 2) = 24` style — needs non-trivial structure |

**Why this task?** Game of 24 is the canonical *catastrophic early-commit* problem. If your first operation is wrong, the chain is dead — you cannot recover by writing more text. CoT has no way to back up. ToT can.

In [ ]:
# -- Deterministic verifier: does a string expression evaluate to 24? ------
# (Bonus: in real-world ToT, a deterministic verifier like this is gold.)

def extract_expression(text: str) -> Optional[str]:
    """Pull the last arithmetic expression out of a model reply."""
    # Strip everything that isn't math, then find a clean expression.
    # Look for an explicit "= 24" line first.
    m = re.search(r"([0-9+\-*/().\s]+?)\s*=\s*24\b", text)
    if m:
        return m.group(1).strip()
    # Fallback: last line that contains only math chars.
    for line in reversed(text.strip().splitlines()):
        line = line.strip().rstrip("=24").strip()
        if line and all(c in "0123456789+-*/(). " for c in line):
            return line
    return None


def uses_exact_numbers(expr: str, numbers: list[int]) -> bool:
    """Check expr uses each number in `numbers` exactly once (as a multiset)."""
    found = list(map(int, re.findall(r"\d+", expr)))
    return sorted(found) == sorted(numbers)


def verify_24(expr: Optional[str], numbers: list[int]) -> bool:
    if not expr:
        return False
    if not uses_exact_numbers(expr, numbers):
        return False
    try:
        return abs(eval(expr) - 24) < 1e-6  # noqa: S307 (sandboxed math)
    except Exception:
        return False


# Quick self-check
assert verify_24("5 * 5 - 5 / 5", [5, 5, 5, 5])
assert not verify_24("5 + 5 + 5 + 5", [5, 5, 5, 5])
print("Verifier OK.")

## Baseline — Zero-shot Chain of Thought

Single linear chain. "Let's think step by step." One commit, no backtracking. Repeat 5 times and count successes.

Expected: the model often picks a bad first operation (e.g. `5 + 5 + ...`) and can't recover.

**Temperature note:** CoT baseline runs at `TEMPERATURE` (default `0.0`) for reproducibility — the failure mode is structural, not sampling-noise. ToT's **propose** step below intentionally lifts temperature to `≥ 0.5` because *search needs diversity*: identical proposals waste the beam.

In [ ]:
# -- Baseline: Zero-shot CoT, 5 trials -------------------------------------

COT_PROMPT = (
    "Use the numbers {nums} exactly once each, with + - * / and parentheses, "
    "to make 24. Let's think step by step. End your answer with a single line "
    "of the form `<expression> = 24`."
)


@mlflow.trace(
    name="cot_solve",
    span_type=SpanType.LLM,
    attributes={"technique": "zero_shot_cot"},
)
def cot_solve(numbers: list[int], temperature: float = TEMPERATURE) -> tuple[str, Optional[str]]:
    prompt = COT_PROMPT.format(nums=numbers)
    # Tape-recorder: capture the prompt and params we sent into this span.
    span = mlflow.get_current_active_span()
    if span:
        span.set_inputs({"prompt": prompt, "params": {"model": MODEL, "temperature": temperature, "numbers": numbers}})
    t0 = time.time()
    if USE_MOCK:
        # Deterministic mock: pretend the model wrote a wrong answer.
        text = "Step 1: try 5+5+5+5. That gives 20, not 24. = 20"
        cost = 0.0
    else:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_tokens=300,
        )
        text = resp.choices[0].message.content
        cost = price_of(MODEL, resp.usage)
    latency_ms = (time.time() - t0) * 1000
    tag_cost_latency(latency_ms, cost)
    expr = extract_expression(text)
    if span:
        span.set_outputs({"text": text, "expression": expr})
    return text, expr


def run_cot_trials(numbers: list[int], n_trials: int = 5) -> pd.DataFrame:
    rows = []
    for i in range(n_trials):
        text, expr = cot_solve(numbers)
        ok = verify_24(expr, numbers)
        rows.append({"trial": i + 1, "expression": expr, "verified": ok})
    return pd.DataFrame(rows)


cot_df = run_cot_trials([5, 5, 5, 5], n_trials=5)
print(cot_df.to_string(index=False))
print()
print(f"LIVE (illustrative, n=5): CoT success rate on [5,5,5,5]: {cot_df['verified'].mean():.0%}")
print("(Headline 4% → 74% from Yao et al. 2023 is the citation, not this run.)")

## What just happened

Even on the *easy* `[5, 5, 5, 5]` puzzle, plain CoT often fails. Why?

- CoT commits left-to-right. If the first move is `5 + 5`, the remaining numbers can't reach 24 — but the model has already invested in that path.
- There's no scoring mid-chain. The model doesn't *know* it's on a dead branch until the final answer fails to land.
- Self-Consistency (N independent CoT chains, vote) helps a little, but the underlying distribution is still skewed toward "obvious-but-wrong" openings.

This is the **catastrophic early-commit** pattern — the exact regime ToT was designed for. Yao et al. 2023 measured GPT-4 at **4% (CoT)** vs **74% (ToT, b=5)** on Game of 24.

## Tree of Thoughts — propose, evaluate, search

Three primitives:

1. **`propose(state, k)`** — ask the model for `k` next-step expressions (one operation each).
2. **`evaluate(state)`** — ask the model whether the *remaining* numbers can still reach 24. Map `sure / maybe / impossible → 1.0 / 0.5 / 0.0`.
3. **`tot_bfs(...)`** — BFS with beam pruning. At each level: expand every frontier node, score every child, keep top `beam`.

Logged with MLflow: every expansion is a `CHAIN` span; every evaluation is an `LLM` span; the full tree is dropped as a table artefact via `mlflow.log_table`.

In [ ]:
# -- propose / evaluate primitives -----------------------------------------

PROPOSE_PROMPT = (
    "You are solving the Game of 24. Current numbers available: {nums}.\n"
    "Goal: combine them with + - * / and parentheses to reach 24.\n\n"
    "Propose {k} distinct *next* single-operation moves. Each move picks two "
    "numbers from the current set and combines them, leaving a smaller set.\n"
    "Output format: one move per line, like `a OP b = c (remaining: x, y, c)`.\n"
    "Be concrete. No prose."
)

EVAL_PROMPT = (
    "You are evaluating a Game-of-24 partial state.\n"
    "Remaining numbers after some moves: {nums}.\n"
    "Can these be combined with + - * / and parentheses to reach exactly 24?\n"
    "Respond with EXACTLY one word: sure, maybe, or impossible."
)

_VERDICT_SCORES = {"sure": 1.0, "maybe": 0.5, "impossible": 0.0}

# ToT propose-step temperature. Search needs DIVERSITY: identical proposals waste the beam.
# Keep this > 0.5 even when TEMPERATURE = 0.0.
PROPOSE_TEMPERATURE = max(0.8, TEMPERATURE if TEMPERATURE >= 0.5 else 0.8)


def _parse_move(line: str) -> Optional[tuple[str, list[float]]]:
    """Parse `a OP b = c (remaining: x, y, c)` -> (expression-so-far, remaining_nums)."""
    m = re.match(r"\s*(.+?)\s*\(remaining:\s*([^)]*)\)", line)
    if not m:
        return None
    move = m.group(1).strip()
    try:
        remaining = [float(x.strip()) for x in m.group(2).split(",") if x.strip()]
    except ValueError:
        return None
    return move, remaining


def propose(state: dict, k: int = 3) -> list[dict]:
    """Return up to k candidate child states. state = {'nums': [...], 'moves': [...]}"""
    with mlflow.start_span(name="propose", span_type=SpanType.LLM) as sp:
        prompt = PROPOSE_PROMPT.format(nums=state["nums"], k=k)
        # Tape-recorder: record exactly what we asked for.
        sp.set_inputs({"prompt": prompt, "params": {"model": MODEL, "k": k, "state_nums": state["nums"],
                                                      "temperature": PROPOSE_TEMPERATURE}})

        t0 = time.time()
        if USE_MOCK:
            # Deterministic mock: hand back a known path for [5,5,5,5]
            mocks = [
                {"move": "5 * 5 = 25",  "nums": [25.0, 5.0, 5.0]},
                {"move": "5 + 5 = 10",  "nums": [10.0, 5.0, 5.0]},
                {"move": "5 / 5 = 1",   "nums": [1.0, 5.0, 5.0]},
            ][:k]
            children = [{"nums": m["nums"], "moves": state["moves"] + [m["move"]]} for m in mocks]
            latency_ms = (time.time() - t0) * 1000
            sp.set_attribute("latency_ms", latency_ms)
            sp.set_attribute("cost_usd", 0.0)
            sp.set_outputs({"text": "[MOCK]", "children": children})
            sp.set_attribute("n_children", len(children))
            return children

        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=PROPOSE_TEMPERATURE,   # diversity ≥ 0.5 — search needs varied proposals
            max_tokens=200,
        )
        latency_ms = (time.time() - t0) * 1000
        cost = price_of(MODEL, resp.usage)
        sp.set_attribute("latency_ms", latency_ms)
        sp.set_attribute("cost_usd", cost)
        text = resp.choices[0].message.content
        children = []
        for line in text.splitlines():
            parsed = _parse_move(line)
            if parsed is None:
                continue
            move, remaining = parsed
            children.append({"nums": remaining, "moves": state["moves"] + [move]})
            if len(children) >= k:
                break
        sp.set_outputs({"text": text, "children": children})
        sp.set_attribute("n_children", len(children))
        return children


def evaluate(state: dict) -> float:
    """Score how reachable 24 is from this state. 1.0=sure, 0.5=maybe, 0.0=impossible."""
    nums = state["nums"]
    with mlflow.start_span(name="evaluate", span_type=SpanType.LLM) as sp:
        sp.set_inputs({"prompt": EVAL_PROMPT.format(nums=nums) if len(nums) > 1 else "(deterministic terminal check)",
                       "params": {"model": MODEL, "nums": nums, "temperature": 0.0}})

        # Cheap deterministic check: if one number left, score by exact match.
        if len(nums) == 1:
            score = 1.0 if abs(nums[0] - 24) < 1e-6 else 0.0
            sp.set_outputs({"text": "terminal", "score": score})
            sp.set_attribute("score", score)
            sp.set_attribute("latency_ms", 0.0)
            sp.set_attribute("cost_usd", 0.0)
            return score

        t0 = time.time()
        if USE_MOCK:
            # Mock: prefer states whose product/sum can still reach 24-ish.
            score = 0.5 if 24 in nums or 25 in nums else 0.3
            latency_ms = (time.time() - t0) * 1000
            sp.set_attribute("latency_ms", latency_ms)
            sp.set_attribute("cost_usd", 0.0)
            sp.set_outputs({"text": "[MOCK]", "score": score})
            sp.set_attribute("score", score)
            return score

        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": EVAL_PROMPT.format(nums=nums)}],
            temperature=0.0,   # evaluator MUST be deterministic for stable scoring
            max_tokens=5,
        )
        latency_ms = (time.time() - t0) * 1000
        cost = price_of(MODEL, resp.usage)
        sp.set_attribute("latency_ms", latency_ms)
        sp.set_attribute("cost_usd", cost)
        text = resp.choices[0].message.content or ""
        verdict = text.strip().lower().split()[0] if text else ""
        score = _VERDICT_SCORES.get(verdict, 0.0)
        sp.set_outputs({"text": text, "verdict": verdict, "score": score})
        sp.set_attribute("score", score)
        return score


print("propose / evaluate ready.")
print(f"  propose temperature  = {PROPOSE_TEMPERATURE}  (≥ 0.5 — search needs diversity)")
print(f"  evaluate temperature = 0.0  (deterministic scoring)")

In [ ]:
# -- tot_bfs: beam search with full MLflow tracing -------------------------

@mlflow.trace(
    name="tot_solve",
    span_type=SpanType.CHAIN,
    attributes={"beam": 3, "depth": 4, "k": 3},
)
def tot_bfs(numbers: list[int], beam: int = 3, depth: int = 4, k: int = 3) -> dict:
    """Beam-search Tree of Thoughts. Returns the best terminal/frontier node found."""
    # Tape-recorder: record the search request on the parent CHAIN span.
    root_span = mlflow.get_current_active_span()
    if root_span:
        root_span.set_inputs({"prompt": f"Solve Game of 24 for {numbers}",
                              "params": {"model": MODEL, "beam": beam, "depth": depth, "k": k, "numbers": numbers}})

    root = {"id": "r", "nums": [float(n) for n in numbers], "moves": [], "score": 0.0,
            "parent": None, "depth": 0, "pruned": False}
    frontier = [root]
    all_nodes: list[dict] = [root]
    best = root

    for d in range(depth):
        next_frontier: list[dict] = []

        for node in frontier:
            with mlflow.start_span(
                name=f"expand_{node['id']}",
                span_type=SpanType.CHAIN,
            ) as sp:
                # Tape-recorder: what came into this expansion.
                sp.set_inputs({"prompt": f"Expand node {node['id']} at depth {d}",
                               "params": {"node_id": node["id"], "nums": node["nums"], "moves": node["moves"], "k": k}})
                sp.set_attribute("depth", d)
                sp.set_attribute("parent_score", node["score"])
                sp.set_attribute("parent_nums", str(node["nums"]))

                children = propose({"nums": node["nums"], "moves": node["moves"]}, k=k)
                sp.set_attribute("n_children", len(children))

                scored_children = []
                for i, child in enumerate(children):
                    nid = f"{node['id']}.{i}"
                    score = evaluate(child)
                    cn = {
                        "id": nid,
                        "nums": child["nums"],
                        "moves": child["moves"],
                        "score": score,
                        "parent": node["id"],
                        "depth": d + 1,
                        "pruned": False,
                    }
                    next_frontier.append(cn)
                    all_nodes.append(cn)
                    scored_children.append({"id": nid, "nums": cn["nums"], "score": score})
                    if score > best["score"]:
                        best = cn
                    mlflow.log_metric(f"node_score.{nid}", score, step=d)

                # Tape-recorder: what came out of this expansion.
                sp.set_outputs({"text": f"{len(children)} children scored",
                                "children": scored_children})

        # Beam prune.
        kept = sorted(next_frontier, key=lambda x: -x["score"])[:beam]
        kept_ids = {n["id"] for n in kept}
        for n in next_frontier:
            if n["id"] not in kept_ids:
                n["pruned"] = True
        frontier = kept
        mlflow.log_metric("frontier_size", len(frontier), step=d)
        mlflow.log_metric("pruned_count", len(next_frontier) - len(frontier), step=d)

        # Early stop if any terminal node hit 24.
        if any(len(n["nums"]) == 1 and abs(n["nums"][0] - 24) < 1e-6 for n in frontier):
            break

    # Log the entire search tree as a browsable table artefact.
    tree_table = {
        "id":     [n["id"]              for n in all_nodes],
        "parent": [n["parent"]          for n in all_nodes],
        "depth":  [n["depth"]           for n in all_nodes],
        "score":  [n["score"]           for n in all_nodes],
        "pruned": [n["pruned"]          for n in all_nodes],
        "nums":   [str(n["nums"])       for n in all_nodes],
        "moves":  [" | ".join(n["moves"]) for n in all_nodes],
    }
    mlflow.log_table(data=tree_table, artifact_file="search_tree.json")

    # Tag the trace so we can filter past runs by outcome.
    mlflow.update_current_trace(tags={
        "tot_best_score":    f"{best['score']:.3f}",
        "tot_nodes_total":   str(len(all_nodes)),
        "tot_depth_reached": str(best["depth"]),
        "tot_puzzle":        str(numbers),
    })
    # Tape-recorder: final search result on the parent CHAIN span.
    if root_span:
        root_span.set_outputs({"text": f"best={best['id']} score={best['score']:.2f} nums={best['nums']}",
                               "best": best, "n_nodes": len(all_nodes)})
    return best


print("tot_bfs ready.")

In [ ]:
# -- Run ToT on both puzzles -----------------------------------------------

for puzzle in [[5, 5, 5, 5], [2, 7, 8, 8]]:
    print(f"\n=== ToT on {puzzle} ===")
    winner = tot_bfs(puzzle, beam=3, depth=4, k=3)
    print(f"  best score    : {winner['score']:.2f}")
    print(f"  final nums    : {winner['nums']}")
    print(f"  move trail    :")
    for step, move in enumerate(winner["moves"], 1):
        print(f"    {step}. {move}")

## Open the MLflow UI

1. Server already running? Visit **http://127.0.0.1:5000** in your browser.
2. Pick experiment **`session6/demo_01_tot`** → Traces tab.
3. Click into the latest **`tot_solve`** trace.
4. Expand the span tree — you'll see:
   - `tot_solve` (CHAIN) at the root
   - `expand_r`, `expand_r.0`, `expand_r.1`… (CHAIN spans for every BFS expansion)
   - Each expand contains the auto-traced OpenAI `propose` calls + per-child `evaluate` LLM calls
5. **Artifacts tab → `search_tree.json`** opens as a sortable table. Filter `pruned=True` to audit what the beam threw away.
6. **Metrics tab** — `frontier_size`, `pruned_count`, and one `node_score.<id>` series per node, stepped by depth.

Trace tags `tot_best_score`, `tot_nodes_total`, `tot_depth_reached`, `tot_puzzle` are searchable across runs (next cell).

In [ ]:
# -- Query past successful ToT searches -------------------------------------
# (Run this after a few ToT runs have accumulated.)

past = mlflow.search_traces(
    experiment_names=[MLFLOW_EXPERIMENT],
    filter_string="tags.tot_best_score >= '0.8'",
    max_results=5,
)

if len(past) == 0:
    print("No high-scoring traces found yet — run the ToT cell above a few times.")
else:
    cols = [c for c in ["trace_id", "tags.tot_puzzle", "tags.tot_best_score",
                         "tags.tot_nodes_total", "tags.tot_depth_reached"]
            if c in past.columns]
    print(past[cols].to_string(index=False))

## The cost reality — 30–50×

Open any `tot_solve` trace in MLflow and look at the **token usage** at the top of the trace view. Compare to a `cot_solve` trace.

Rough call count:

| Technique | LLM calls per puzzle |
|---|---|
| Zero-shot CoT | **1** |
| ToT (beam=3, depth=4, k=3) | propose: `depth × beam × 1 ≈ 12`; evaluate: `depth × beam × k ≈ 36` → **~48 calls** |

That's the 30–50× multiplier from Yao et al. It's why ToT belongs in **offline / high-stakes** workflows (root-cause analysis, code synthesis pipelines, batch generation) — not in interactive chat where latency and per-call cost matter.

## When does ToT actually help?

The value function is everything. ToT *amplifies* whatever signal your evaluator produces:

- **Good evaluator** (deterministic verifier, or a clear sure/maybe/impossible signal) → ToT prunes confidently and search converges. This is the Game-of-24 regime.
- **Noisy evaluator** (LLM-as-judge on subjective output, std-dev > 0.2 across re-scores) → ToT becomes expensive random sampling.

Cross-reference *Misconception 2* in [`02-tree-of-thoughts.md`](../../03-wiki/02-tree-of-thoughts.md): **“ToT is not just CoT × N. The scorer + pruner is the mechanism. Without a reliable scorer, ToT is worse than Self-Consistency at higher cost.”**

Smell test before reaching for ToT:

1. Write your evaluator.
2. Score the same partial state 5× at temperature 0.7.
3. If std-dev > 0.2 (on 0–1 scale) — stop. Use Self-Consistency or chain with deterministic validators.

## Takeaways

- **Search beats sampling for branching problems.** When the first commit can be catastrophically wrong, propose + evaluate + prune wins over N independent chains.
- **A reliable value function is non-negotiable.** ToT amplifies signal *and* noise. If you cannot score a partial state, ToT will burn money.
- **Cost is the trade.** Plan for 30–50× the tokens of a single CoT call. Use cheap models for `propose`, strong models for `evaluate` when possible.
- **MLflow tree view is the diagnostic.** Span tree shows *how* the search went; `search_tree.json` shows *what* was pruned. Together they make ToT debuggable instead of mystical.

Next demo: chaining with validators — the same "score-and-prune" idea applied to *deterministic* gates.

## ⚠️ Reasoning model caveat

This notebook assumes a **non-reasoning** base model. On `o1` / `o3` / Claude Extended Thinking / Gemini Thinking:
- Drop the CoT / ToT / Plan-and-Solve scaffolding — the model does it internally
- Use `developer` role instead of `system` on `o1-2024-12-17+`
- Never pass OpenAI `reasoning_effort` and Gemini `thinking_level` through the same OpenAI-compatible adapter
- See Block 5 of the slides for what's redundant vs what survives

## 🎥 Replay every input + output in MLflow

Every prompt and every response from this demo is now in MLflow. Open the UI and click any trace to see the full I/O log.

In [ ]:
import urllib.parse
import IPython.display as D

ui = MLFLOW_TRACKING_URI.rstrip("/")
exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
exp_id = exp.experiment_id if exp else "0"
link = f"{ui}/#/experiments/{exp_id}?searchFilter=&orderByKey=attributes.start_time&orderByAsc=false"
D.display(D.Markdown(f"**Open the experiment in MLflow:** [{link}]({link})"))

# Recent traces (last 5)
recent = mlflow.search_traces(experiment_ids=[exp_id], max_results=5)
recent[["request_id", "timestamp_ms", "execution_time_ms", "tags"]] if not recent.empty else "No traces yet — run the cells above."